# Project 2 starter: Unclaimed Property at Silicone Motors

ACCTG 6155, Fall 2026. Due Sunday, October 25, 11:59 PM MT.

**First, File > Save a copy in Drive, and work in your copy.**

This notebook is optional. It gives you:

- a load cell for every table,
- a stub for each analysis step in 4.1 of the guide,
- a worked example of the look-back and the breakage estimate on the same 120 practice accounts the [return-rate trainer](https://seanmccaman.com/acctg6155/2026-fall/project-2/interactive/island_trainer.html) uses, with cells that check your code before you run it on the full data,
- a last section that lays out every number in 3.4 for your data support file.

The guide, the counsel memo and the data dictionary are on the [Project 2 page](https://seanmccaman.com/acctg6155/2026-fall/project-2/). The memo is the source for every rule. Nothing in this notebook tells you what the memo says, so read it before section 1.

When you are done, run every cell. This notebook, with its outputs showing, can be your data support file (section 9). The deck and the one-pager are separate files.

## 0. Load the data

The next cell loads every table from the course GitHub repository. If you work in Jupyter on your own machine and unzipped `p2_csv.zip` next to this notebook, it loads from that folder instead. Money is in integer cents. Keep it in cents until you report, because Excel, pandas and SQLite round differently.

In [ ]:
import os
import pandas as pd
from IPython.display import display

BUILD_ID = "c14c2737ec6d"   # the data build this notebook expects; the row counts below are the check
RAW = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/project-2/data/csv/"
LOCAL = "p2_csv"          # the unzipped CSV download, if you have it

TABLES = ['drivers', 'wallet_ledger', 'charging_sessions', 'app_login_days', 'reservations', 'reservation_events', 'company_notices', 'due_diligence_letters', 'gl_wallet_control', 'run_params']
EXPECTED_ROWS = {'drivers': 8000, 'wallet_ledger': 57916, 'charging_sessions': 39556, 'app_login_days': 32135, 'reservations': 2104, 'reservation_events': 3048, 'company_notices': 12673, 'due_diligence_letters': 422, 'gl_wallet_control': 270, 'run_params': 1}
TEXT_COLUMNS = ["state_raw", "postal_raw", "country_raw", "address_line1", "city", "entered_by", "gl_account"]


def load(name):
    local = os.path.join(LOCAL, name + ".csv")
    df = pd.read_csv(local if os.path.exists(local) else RAW + name + ".csv",
                     dtype={c: str for c in TEXT_COLUMNS})
    for c in df.columns:
        if c.endswith(("_on", "_date", "_end", "_open", "_close")):
            df[c] = pd.to_datetime(df[c])
    return df


data = {t: load(t) for t in TABLES}
for t in TABLES:
    flag = "" if len(data[t]) == EXPECTED_ROWS[t] else "   <- not the expected count; reload from the Project 2 page"
    print(f"{t:24s}{len(data[t]):>8,d} rows{flag}")

drivers      = data["drivers"]
ledger       = data["wallet_ledger"]
sessions     = data["charging_sessions"]
logins       = data["app_login_days"]
reservations = data["reservations"]
res_events   = data["reservation_events"]
notices      = data["company_notices"]
letters      = data["due_diligence_letters"]
gl           = data["gl_wallet_control"]
params       = data["run_params"].iloc[0]

CUTOFF = params["cutoff_date"]
print("\nCutoff:", CUTOFF.date(), "| expected build:", BUILD_ID)


def dollars(cents):
    """For display only. Do the arithmetic in cents."""
    return f"${cents / 100:,.2f}"

In [ ]:
ledger.head()

Prefer SQL? The same tables are in one SQLite file. This cell uses your copy if one is next to the notebook, downloads it otherwise, and runs a query. After that you can use `q("...")` for the rest of the project. If you work in pandas only, you can skip it.

In [ ]:
import sqlite3
import urllib.request

DB_FILE = next((p for p in ["p2_wallet.sqlite", os.path.join("data", "p2_wallet.sqlite")] if os.path.exists(p)), None)
if DB_FILE is None:
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/project-2/data/p2_wallet.sqlite", "p2_wallet.sqlite")
        DB_FILE = "p2_wallet.sqlite"
    except Exception as e:
        print(f"Could not download the database ({e}). Download it from the Project 2 page and put it "
              "next to this notebook, or skip this cell and work in pandas.")

if DB_FILE:
    db = sqlite3.connect(DB_FILE)

    def q(sql):
        return pd.read_sql(sql, db)

    display(q("SELECT txn_type, COUNT(*) AS n FROM wallet_ledger GROUP BY txn_type ORDER BY n DESC"))

## 1. The rule table (4.1 step 1)

Build it from the counsel memo before you write any analysis code, and check every cell against the memo, not against an AI. Schedule A lists every state by its full name. Your table needs a row for each of the nine states where the company has drivers, keyed by the two-letter code the data uses. Add a column for each rule the memo sets by state. Rules that apply to every state go in the cell after, as plain Python values, so the code below uses them instead of retyped numbers.

The memo has a footnote that changes a number.

In [ ]:
STATES = ['AZ', 'CA', 'CO', 'DE', 'NV', 'NY', 'TX', 'UT', 'WA']   # the nine states with drivers, as the data codes them, in Schedule A's order
rules = pd.DataFrame(index=pd.Index(STATES, name="state"))

# One column per rule in the memo's schedules. For example:
# rules["wallet_period_years"] = [...]   # nine values, in the order of STATES

rules

In [ ]:
# Memo-wide rules. Fill these in from the memo.
OWNER_CONTACT_CHANNELS = set()   # source_channel values that are the owner's own doing
OWNER_CONTACT_TYPES = set()      # txn_type values that can be owner contact
FALLBACK_STATE = None            # where a balance reports when the memo's priority rule sends it elsewhere

## 2. Rebuild the balances and tie them to the GL (4.1 step 2; 3.4 items 1 to 3)

Sum the ledger per driver as of the cutoff, on the date each transaction happened (`activity_on`), with cash and promotional credit kept apart. The data dictionary lists the transaction types. Then compare your totals with `gl_wallet_control` at June 30.

One of them will not tie. The difference has a cause you can find in the data, and it is a reconciling item to explain, not an error to plug.

In [ ]:
CASH_TYPES = set()    # txn_type values that move the owner's cash
PROMO_TYPES = set()   # txn_type values that move promotional credit

# asof = ledger[ledger["activity_on"] <= CUTOFF]
# cash_by_driver = ...
# promo_by_driver = ...

gl[gl["period_end"] == CUTOFF]

## 3. The owner summary table (4.1 step 3)

Start from `drivers`, because it holds the whole population, and bring everything else in with a left join (`how="left"`), so no owner drops out. The goal is one row per owner with these columns: `driver_id`, `clean_state`, jurisdiction, cash balance, promo balance, last owner wallet transaction, last login, last letter response, last owner contact, and mailable. Some columns fill in here and the rest in section 4. Deposits run their own clocks, so build a second table the same way, one row per reservation.

Use the memo's definition of owner contact and nothing else. Two mistakes cost real money here: entries the company generated, and transactions that were later reversed. The ledger links each reversal to the row it backs out. A refund also fills in `reverses_txn_id`, pointing at the charge it corrects, so read the type: only a reversal erases the row it points to. A reply to a due diligence letter from an earlier cycle is owner contact.

In [ ]:
reversals = ledger[ledger["txn_type"] == "reversal"]
reversals[["txn_id", "driver_id", "activity_on", "amount_cents", "source_channel", "reverses_txn_id"]].head()

In [ ]:
owners = drivers[["driver_id", "state_raw"]].copy()   # one row per owner; bring everything else in with how="left"

# owners["cash_cents"] = ...             # cash balance at the cutoff, from section 2
# owners["promo_cents"] = ...            # promotional credit at the cutoff
# owners["last_wallet_txn"] = ...        # owner wallet transactions only: no company entries, nothing reversed, on or before the cutoff
# owners["last_login"] = ...
# owners["last_letter_response"] = ...   # replies to letters from earlier cycles
# clean_state, jurisdiction, last_contact and mailable come in section 4

assert owners["driver_id"].is_unique     # rerun after every join: a join that adds rows breaks it

deposits = reservations[["reservation_id", "driver_id"]].copy()   # one row per reservation, with its own clock
# deposits["deposit_cents"] = ...        # the deposit still held at the cutoff
# deposits["last_contact"] = ...         # owner-initiated events on that reservation, from res_events
assert deposits["reservation_id"].is_unique

## 4. Clean the state and resolve the jurisdiction (4.1 step 4)

`state_raw` was typed by people. Look at every spelling before you map anything. The first cell prints each value with `repr`, which shows the quotes, so a stray space that a plain print hides is visible. Put the cleaned code in a new `clean_state` column next to `state_raw`. Never overwrite the raw field, so you can sample rows and check your mapping. A join that silently drops the rows it cannot match is the most common way this analysis goes wrong, so end this section with a check that no owner is left without a jurisdiction.

Then finish the owner summary table. Whether a login counts as contact depends on the jurisdiction, so last owner contact comes after it.

In [ ]:
drivers["state_raw"].map(repr).value_counts(dropna=False)

In [ ]:
drivers["country_raw"].value_counts(dropna=False)

In [ ]:
# STATE_MAP = {...}                      # every spelling you saw, to a two-letter code
# owners["clean_state"] = ...            # a new column next to state_raw; never overwrite state_raw
# owners["jurisdiction"] = ...           # the memo's priority rule decides this, not the state column alone
# assert owners["jurisdiction"].notna().all()

# owners["last_contact"] = ...           # the latest owner contact; a login counts only where the jurisdiction's schedule says so
# owners["mailable"] = ...               # whether this cycle's letter can be mailed; the memo says when it cannot

## 5. Apply the rules (4.1 step 4; 3.4 items 4 to 7)

Add the dormancy period to the last contact date and compare the result with the cutoff. An anniversary on the cutoff itself is dormant. Months are calendar months, counted the way Excel's `EDATE` counts them, and `add_months` does that for you: August 31 plus 6 months is the last day of February. Counting days instead (365 for a year) drifts by a day around leap years, and a day is enough to move an owner across the cutoff.

In [ ]:
def add_months(dates, k):
    """Add k calendar months to a Series of dates, the way Excel's EDATE does."""
    return pd.to_datetime(dates) + pd.DateOffset(months=k)


def add_months_each(dates, months):
    """Like add_months, when each row has its own number of months (for example, a period by state)."""
    out = pd.Series(pd.NaT, index=dates.index, dtype="datetime64[ns]")
    for k in pd.unique(months.dropna()):
        rows = months == k
        out[rows] = add_months(dates[rows], int(k))
    return out


add_months(pd.Series(pd.to_datetime(["2023-08-31", "2024-02-29", "2023-06-30"])), 6)

In [ ]:
# dormant wallets: count, and reportable amount by state and in total    (3.4 items 4 and 6)
# dormant deposits: count, and reportable amount by state and in total   (3.4 items 5 and 6)
# due diligence letters: the list of owners (driver_id) who get one this cycle, and the count   (3.4 item 7)
#   The list has no bearing on your other numbers: everything is as of June 30,
#   before any of this year's letters go out.

# Checks worth running as you go: one row per owner (or per reservation) after every join,
# no wallet from an exempt state in the dormant count, and at most one letter per owner.

## 6. The look-back (4.1 step 5; 3.4 item 8), on practice data first

The question is simple. When an owner goes quiet, will they come back on their own? You answer it by looking back. Pick a date with two full years of records after it, June 30, 2024. Find the owners who were quiet on that date. Then count how many of them made an owner wallet transaction in the next year, and in the next two years.

**Start with 100 accounts.** In the practice data, 100 accounts had been quiet for a year or more on June 30, 2024, with money left in their wallets. One year later, 93 of them were still quiet. Two years later, 88 were. So 12 came back within two years, and 7 of those came back in the first year.

**Two groups.** The look-back sorts quiet owners by how long they had been quiet on June 30, 2024, the way an aging schedule sorts receivables. It uses each owner's last owner wallet transaction on or before that date, and only those dates:

- **Quiet 3 years or more:** the last transaction was on or before June 30, 2021.
- **Quiet 1 to 3 years:** the last transaction was after June 30, 2021 and on or before June 30, 2023.

Owners with no transaction by June 30, 2024, and owners with one in the year before it, are in neither group. Owners in every state go in, whatever their balance.

**Came back.** An owner came back within one year with a transaction after June 30, 2024 and on or before June 30, 2025. They came back within two years with one on or before June 30, 2026. Within two years includes the owners who came back in the first year. That gives four rates: for each group, the share that came back within one year and within two years.

**Stopping the clock early.** The one-year count looks at the same owners with half the time to come back. An owner who comes back in the second year still counts as gone, so the one-year rate reads low. A lower return rate means a higher breakage estimate, and breakage is revenue, so stopping the count early overstates it.

The [return-rate trainer](https://seanmccaman.com/acctg6155/2026-fall/project-2/interactive/island_trainer.html) walks through every step on the same practice data: 120 made-up accounts, A001 to A120. Their dates are already filtered. Each one is an owner wallet transaction, once per day. On the real data you build that list yourself, as 3.4 item 8 defines it.

In [ ]:
# The practice data: 120 made-up accounts. Each row is the account, its made-up practice state, its
# wallet cash at June 30, 2026 in cents, and its owner wallet transaction dates, already filtered.
PRACTICE = [
    ("A001", "Alder", 17666, ["2018-07-20", "2018-11-06", "2018-12-28", "2019-08-20", "2024-10-14", "2025-01-08"]),
    ("A002", "Alder", 10687, ["2022-11-09"]),
    ("A003", "Cedar", 3708, ["2023-04-04", "2023-10-08", "2025-06-30", "2025-11-11", "2025-12-24"]),
    ("A004", "Isle", 33974, ["2020-09-02", "2021-02-18", "2021-06-10"]),
    ("A005", "Birch", 26995, ["2020-07-17", "2020-08-19", "2021-03-23", "2021-08-29"]),
    ("A006", "Birch", 17760, ["2021-04-19"]),
    ("A007", "Alder", 10781, ["2022-03-30", "2022-10-13"]),
    ("A008", "Alder", 3802, ["2023-04-29", "2023-10-05", "2024-01-15"]),
    ("A009", "Birch", 34068, ["2019-06-04", "2019-10-23", "2020-01-15", "2020-02-11"]),
    ("A010", "Grove", 27089, ["2021-08-02"]),
    ("A011", "Elm", 17854, ["2019-10-17", "2019-12-21"]),
    ("A012", "Alder", 10875, ["2021-12-09", "2022-05-27", "2022-09-16", "2025-01-30", "2025-04-11", "2025-07-20"]),
    ("A013", "Cedar", 3896, ["2024-07-01", "2024-08-13", "2024-10-24"]),
    ("A014", "Elm", 34162, ["2021-11-02", "2024-07-01", "2024-09-24"]),
    ("A015", "Dune", 27183, ["2020-11-19", "2021-07-06"]),
    ("A016", "Dune", 17948, ["2017-12-09", "2018-01-15", "2018-08-23"]),
    ("A017", "Heath", 10969, ["2021-12-11", "2022-05-01", "2022-07-24", "2022-08-20"]),
    ("A018", "Alder", 1734, ["2021-06-30"]),
    ("A019", "Cedar", 34256, ["2021-08-02", "2021-10-06"]),
    ("A020", "Fern", 27277, ["2022-06-27", "2023-01-14", "2023-06-07"]),
    ("A021", "Fern", 18042, ["2018-09-24", "2019-05-31", "2019-12-09", "2020-04-22"]),
    ("A022", "Alder", 11063, ["2022-07-24"]),
    ("A023", "Cedar", 1828, ["2019-09-10", "2020-03-01"]),
    ("A024", "Isle", 34350, ["2020-12-26", "2021-02-01", "2021-09-09"]),
    ("A025", "Birch", 27371, ["2022-07-25", "2023-03-21", "2023-09-19", "2024-01-22"]),
    ("A026", "Birch", 18136, ["2018-12-24"]),
    ("A027", "Alder", 11157, ["2022-03-22", "2022-06-27"]),
    ("A028", "Alder", 1922, ["2018-03-14", "2018-08-06", "2018-11-02"]),
    ("A029", "Birch", 34444, ["2020-01-15", "2020-09-20", "2021-03-31", "2021-08-13"]),
    ("A030", "Grove", 27465, ["2024-04-30", "2026-01-29", "2026-02-21", "2026-04-14"]),
    ("A031", "Elm", 18230, ["2020-01-31", "2020-08-23"]),
    ("A032", "Alder", 11251, ["2021-07-14", "2021-09-21", "2022-05-31"]),
    ("A033", "Cedar", 2016, ["2019-05-09", "2019-09-03", "2019-11-02", "2020-07-02"]),
    ("A034", "Elm", 34538, ["2021-07-17"]),
    ("A035", "Dune", 27559, ["2023-01-26", "2023-08-09"]),
    ("A036", "Grove", 18324, ["2022-01-10", "2022-06-04", "2022-08-31"]),
    ("A037", "Heath", 11345, ["2021-02-26", "2021-04-08", "2021-11-18", "2022-05-04"]),
    ("A038", "Cedar", 2110, ["2019-03-05"]),
    ("A039", "Cedar", 34632, ["2022-11-25", "2023-06-18"]),
    ("A040", "Fern", 27653, ["2025-11-24", "2026-01-15"]),
    ("A041", "Fern", 18418, ["2021-06-10", "2021-10-05", "2021-12-04", "2022-08-04"]),
    ("A042", "Alder", 11439, ["2022-04-07"]),
    ("A043", "Cedar", 2204, ["2020-08-21", "2020-11-02"]),
    ("A044", "Isle", 34726, ["2022-07-29", "2023-01-22", "2023-05-22"]),
    ("A045", "Grove", 25491, ["2019-04-26", "2019-12-07", "2020-05-23", "2020-09-11"]),
    ("A046", "Birch", 18512, ["2022-07-08"]),
    ("A047", "Birch", 11533, ["2024-04-28", "2024-06-30"]),
    ("A048", "Alder", 2298, ["2018-10-06", "2018-11-20", "2019-07-06"]),
    ("A049", "Birch", 34820, ["2022-07-23", "2022-12-19", "2023-03-21", "2023-04-25"]),
    ("A050", "Grove", 25585, ["2019-05-15"]),
    ("A051", "Elm", 18606, ["2022-03-30", "2022-06-11"]),
    ("A052", "Alder", 11627, ["2022-12-06", "2023-01-10", "2023-08-16"]),
    ("A053", "Cedar", 2392, ["2020-03-10", "2020-03-27", "2020-10-13", "2021-03-05"]),
    ("A054", "Elm", 34914, ["2023-03-29"]),
    ("A055", "Dune", 25679, ["2020-07-15", "2021-01-12"]),
    ("A056", "Dune", 18700, ["2021-08-15", "2021-09-29", "2022-05-15"]),
    ("A057", "Heath", 11721, ["2022-05-02", "2023-01-04", "2023-07-13", "2023-11-23", "2024-09-29", "2024-10-31", "2024-12-31"]),
    ("A058", "Cedar", 2486, ["2021-07-01"]),
    ("A059", "Cedar", 35008, ["2022-11-17", "2023-03-02"]),
    ("A060", "Fern", 25773, ["2019-01-09", "2019-06-11", "2019-09-15"]),
    ("A061", "Fern", 18794, ["2021-04-23", "2021-05-10", "2021-11-26", "2022-04-18"]),
    ("A062", "Alder", 11815, ["2024-11-17", "2025-02-11"]),
    ("A063", "Cedar", 2580, ["2022-12-03", "2023-06-02"]),
    ("A064", "Isle", 35102, ["2022-10-29", "2023-01-14", "2023-02-03"]),
    ("A065", "Birch", 25867, ["2020-02-26", "2020-06-30", "2020-09-06", "2021-05-15"]),
    ("A066", "Birch", 18888, ["2022-03-22"]),
    ("A067", "Alder", 9653, ["2021-02-03", "2021-03-24"]),
    ("A068", "Alder", 2674, ["2022-08-30", "2023-01-30", "2023-05-06"]),
    ("A069", "Birch", 35196, ["2021-10-08", "2021-11-26", "2022-07-16", "2023-01-07", "2026-06-30"]),
    ("A070", "Grove", 25961, ["2020-01-16"]),
    ("A071", "Elm", 18982, ["2021-07-25", "2022-02-23", "2025-04-16", "2025-06-19"]),
    ("A072", "Alder", 9747, ["2019-04-14", "2019-05-05", "2019-11-25"]),
    ("A073", "Cedar", 2768, ["2022-01-20", "2022-05-25", "2022-08-01", "2023-04-09"]),
    ("A074", "Elm", 35290, ["2023-11-30"]),
    ("A075", "Dune", 26055, ["2018-06-29", "2018-09-18"]),
    ("A076", "Dune", 19076, ["2021-03-20", "2021-09-21", "2022-01-27"]),
    ("A077", "Heath", 9841, ["2017-02-15", "2017-10-06", "2018-03-31", "2018-07-28"]),
    ("A078", "Cedar", 2862, ["2023-03-13"]),
    ("A079", "Cedar", 35384, ["2023-12-28", "2024-03-08"]),
    ("A080", "Fern", 26149, ["2019-08-03", "2019-09-25", "2020-05-18"]),
    ("A081", "Fern", 19170, ["2021-03-06", "2021-08-10", "2021-11-18", "2021-12-31"]),
    ("A082", "Alder", 9935, ["2020-03-27"]),
    ("A083", "Cedar", 2956, ["2022-11-25", "2023-02-14"]),
    ("A084", "Isle", 35478, ["2023-09-20", "2023-11-02", "2024-06-15", "2025-04-30", "2025-06-10", "2025-08-19"]),
    ("A085", "Birch", 26243, ["2020-10-21", "2021-06-11", "2021-12-04", "2022-04-02"]),
    ("A086", "Birch", 19264, ["2021-12-04"]),
    ("A087", "Alder", 10029, ["2018-05-23", "2018-11-28"]),
    ("A088", "Alder", 3050, ["2022-04-04", "2022-05-27", "2023-01-18"]),
    ("A089", "Birch", 35572, ["2026-06-01"]),
    ("A090", "Grove", 26337, ["2022-03-06"]),
    ("A091", "Elm", 19358, ["2021-07-17", "2021-11-07"]),
    ("A092", "Alder", 10123, ["2019-11-06", "2020-04-15", "2020-07-28"]),
    ("A093", "Cedar", 3144, ["2021-12-03", "2021-12-28", "2022-07-24", "2022-12-22"]),
    ("A094", "Elm", 33410, ["2020-06-06"]),
    ("A095", "Dune", 26431, ["2021-08-02", "2022-02-07"]),
    ("A096", "Dune", 19452, ["2023-07-14", "2023-12-12", "2024-03-15", "2024-11-29", "2025-04-03", "2025-05-07"]),
    ("A097", "Heath", 10217, ["2018-08-15", "2018-12-26", "2019-03-12", "2019-03-31", "2026-02-20", "2026-03-30"]),
    ("A098", "Cedar", 3238, ["2022-11-25"]),
    ("A099", "Cedar", 33504, ["2018-12-12", "2019-02-07"]),
    ("A100", "Fern", 26525, ["2021-04-21", "2021-09-29", "2022-01-11", "2024-11-10"]),
    ("A101", "Fern", 19546, ["2023-04-11", "2023-08-12", "2023-10-17", "2024-06-22"]),
    ("A102", "Alder", 10311, ["2020-11-28"]),
    ("A103", "Alder", 3332, ["2022-03-22", "2022-10-29", "2025-06-30"]),
    ("A104", "Isle", 33598, ["2020-02-09", "2020-03-09", "2020-10-07"]),
    ("A105", "Birch", 26619, ["2021-05-01", "2021-09-11", "2021-11-26", "2021-12-15"]),
    ("A106", "Birch", 19640, ["2023-10-01"]),
    ("A107", "Birch", 10405, ["2023-05-04", "2023-06-30"]),
    ("A108", "Alder", 3426, ["2021-11-07", "2022-05-19", "2022-10-02", "2025-09-25", "2025-10-18", "2025-12-09"]),
    ("A109", "Birch", 33692, ["2017-12-05", "2018-08-03", "2019-02-03", "2019-06-10", "2025-07-01", "2025-10-31"]),
    ("A110", "Grove", 26713, ["2021-11-18"]),
    ("A111", "Elm", 19734, ["2025-05-01", "2025-09-07", "2025-10-15"]),
    ("A112", "Alder", 10499, ["2022-05-06", "2022-06-04", "2023-01-02"]),
    ("A113", "Cedar", 3520, ["2021-10-16", "2022-03-30", "2022-07-16", "2022-09-05", "2026-01-12", "2026-05-10"]),
    ("A114", "Elm", 33786, ["2021-02-07"]),
    ("A115", "Dune", 26807, ["2021-07-25", "2021-10-22"]),
    ("A116", "Dune", 17572, ["2020-05-14", "2020-09-28", "2020-12-17"]),
    ("A117", "Heath", 10593, ["2021-06-02", "2022-01-29", "2022-08-01", "2022-12-06", "2024-08-14", "2024-09-09", "2024-11-03"]),
    ("A118", "Alder", 3614, ["2023-07-01"]),
    ("A119", "Cedar", 33880, ["2019-03-28", "2019-10-11"]),
    ("A120", "Fern", 26901, ["2020-11-24", "2021-01-24", "2021-09-25"]),
]
practice_owners = pd.DataFrame([a[:3] for a in PRACTICE], columns=["owner", "state", "cash_cents"])
practice_txns = pd.DataFrame([(a[0], d) for a in PRACTICE for d in a[3]], columns=["owner", "activity_on"])
practice_txns["activity_on"] = pd.to_datetime(practice_txns["activity_on"])
print(f"{len(practice_owners)} accounts, {len(practice_txns)} transactions")
practice_txns.head()

**The worked example.** The next cells build the look-back on the practice data in four steps. Before you run them, look at the 9 accounts below. Each one sits on a boundary in the rules, and "on or before" decides most of them. The second cell shows the same accounts, so you can check that the code agrees.

| Account | Last transaction on or before June 30, 2024 | First transaction after it | Group | Back within 1 year | Back within 2 years | Why it is here |
|---|---|---|---|---|---|---|
| A018 | 2021-06-30 | none | quiet 3 years or more | No | No | Last transaction on June 30, 2021 itself. On or before counts, so quiet 3 years or more. |
| A058 | 2021-07-01 | none | quiet 1 to 3 years | No | No | One day later, so quiet 1 to 3 years. |
| A107 | 2023-06-30 | none | quiet 1 to 3 years | No | No | Last transaction on June 30, 2023 itself. Still quiet 1 to 3 years. |
| A118 | 2023-07-01 | none | neither | not counted | not counted | One day later. A transaction in the year before June 30, 2024 puts it in neither group. |
| A047 | 2024-06-30 | none | neither | not counted | not counted | A transaction on the look-back date itself. Neither group. |
| A013 | none | 2024-07-01 | neither | not counted | not counted | No transaction until July 1, 2024. Neither group. |
| A103 | 2022-10-29 | 2025-06-30 | quiet 1 to 3 years | Yes | Yes | Back on June 30, 2025. On or before counts, so within one year. |
| A109 | 2019-06-10 | 2025-07-01 | quiet 3 years or more | No | Yes | Back on July 1, 2025, in the second year. Within two years, not within one. |
| A069 | 2023-01-07 | 2026-06-30 | quiet 1 to 3 years | No | Yes | Back on June 30, 2026, the last day that counts. Within two years. |

In [ ]:
LOOKBACK = pd.Timestamp("2024-06-30")          # the look-back date
QUIET_3_THROUGH = pd.Timestamp("2021-06-30")   # quiet 3 years or more: last transaction on or before this
QUIET_1_THROUGH = pd.Timestamp("2023-06-30")   # quiet 1 to 3 years: after QUIET_3_THROUGH, on or before this
ONE_YEAR = pd.Timestamp("2025-06-30")          # back within one year: after LOOKBACK, on or before this
TWO_YEARS = pd.Timestamp("2026-06-30")         # back within two years: after LOOKBACK, on or before this
GROUPS = ["quiet 3 years or more", "quiet 1 to 3 years"]

In [ ]:
pw = practice_owners[["owner"]].copy()   # one row per account

# Step 1. Each account's last transaction on or before the look-back date, from those dates only.
before = practice_txns[practice_txns["activity_on"] <= LOOKBACK]
pw["last_by_lookback"] = pw["owner"].map(before.groupby("owner")["activity_on"].max())

# Step 2. The group, from that one date. An account with no date there is in neither group.
last = pw["last_by_lookback"]
pw["group"] = "neither"
pw.loc[last <= QUIET_3_THROUGH, "group"] = GROUPS[0]
pw.loc[(last > QUIET_3_THROUGH) & (last <= QUIET_1_THROUGH), "group"] = GROUPS[1]

# Step 3. Came back: the first transaction after the look-back date. An empty date never counts.
after = practice_txns[practice_txns["activity_on"] > LOOKBACK]
pw["first_after"] = pw["owner"].map(after.groupby("owner")["activity_on"].min())
pw["back_1y"] = pw["first_after"] <= ONE_YEAR
pw["back_2y"] = pw["first_after"] <= TWO_YEARS

pw.set_index("owner").loc[['A018', 'A058', 'A107', 'A118', 'A047', 'A013', 'A103', 'A109', 'A069']]

In [ ]:
# Of the 100 quiet accounts, how many are still quiet x days after June 30, 2024? Change x and rerun.
x = 365
quiet = pw[pw["group"].isin(GROUPS)]
days_to_return = (quiet["first_after"] - LOOKBACK).dt.days
still_quiet = int((days_to_return.isna() | (days_to_return > x)).sum())
print(f"Of {len(quiet)} quiet accounts, {still_quiet} are still quiet {x} days after June 30, 2024.")

In [ ]:
# Step 4. The four rates: for each group, the share that came back within one year and within two years.
practice_rates = (pw[pw["group"].isin(GROUPS)]
                  .groupby("group")
                  .agg(owners=("owner", "size"), back_1y=("back_1y", "sum"), back_2y=("back_2y", "sum"))
                  .reindex(GROUPS))
practice_rates["rate_1y"] = practice_rates["back_1y"] / practice_rates["owners"]
practice_rates["rate_2y"] = practice_rates["back_2y"] / practice_rates["owners"]
practice_rates.round(4)

**Your turn.** Put the steps in a function, so the same steps run on the real data. The worked example above is one way to write it, and you can start from it. Then run the check. It runs your function on the practice data and compares the result with the answers. If it disagrees, compare your steps with the worked example, or work the boundary accounts above by hand.

In [ ]:
def lookback_rates(txns):
    """
    txns: one row per owner wallet transaction, with columns owner and activity_on.

    Return a DataFrame with one row per group, indexed by the names in GROUPS,
    and the columns owners, back_1y, back_2y, rate_1y and rate_2y.
    Keep the rates unrounded.
    """
    return None   # your code

In [ ]:
# ===== RUN THIS to check lookback_rates on the practice data. You do not edit this cell. =====
EXPECTED_LOOKBACK = {'quiet 3 years or more': {'owners': 40, 'back_1y': 1, 'back_2y': 3},
 'quiet 1 to 3 years': {'owners': 60, 'back_1y': 6, 'back_2y': 9}}
LOOKBACK_HINTS = {
    "owners": "Group each account by its last transaction on or before June 30, 2024, using only the dates on or before it. On or before June 30, 2021 is quiet 3 years or more. After that, and on or before June 30, 2023, is quiet 1 to 3 years.",
    "back_1y": "Within one year means a transaction after June 30, 2024 and on or before June 30, 2025.",
    "back_2y": "Within two years means a transaction after June 30, 2024 and on or before June 30, 2026. It includes the accounts that came back in the first year.",
    "rate": "Each rate is the number that came back divided by the owners in the group, unrounded."
}


def as_number(v):
    """v as a float when it is a number, else None."""
    if isinstance(v, bool):
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if f != f else f


def show(v):
    x = as_number(v)
    if x is None:
        return repr(v)
    return f"{int(x)}" if x == int(x) else f"{x:.4f}"


def report(results, done_message):
    done = sum(r[1] == "done" for r in results)
    print(f"{done} of {len(results)} checks done\n")
    for name, status, hint in results:
        print(f"  {status:8s} {name}" + (f"   ({hint})" if hint else ""))
    if done == len(results):
        print("\n" + done_message)


def check_lookback():
    results = []

    def record(name, ok, hint=""):
        results.append((name, "done" if ok else "not yet", "" if ok else hint))

    fields = ["owners", "back_1y", "back_2y", "rate_1y", "rate_2y"]
    table, waiting = None, None
    try:
        table = lookback_rates(practice_txns.copy())
        if table is None:
            waiting = "it returns None. Write it first."
    except Exception as e:
        waiting = f"it raised {type(e).__name__}: {e}"
    if table is not None and waiting is None:
        if isinstance(table, pd.DataFrame) and "group" in table.columns:
            table = table.set_index("group")
        if not (isinstance(table, pd.DataFrame) and set(fields) <= set(table.columns)
                and all(g in table.index for g in EXPECTED_LOOKBACK)):
            waiting = f"it should return a DataFrame indexed by {list(EXPECTED_LOOKBACK)}, with the columns {fields}"
    record("lookback_rates returns one row per group", waiting is None, waiting or "")

    for g, want in EXPECTED_LOOKBACK.items():
        if waiting:
            for f in fields:
                record(f"{g}: {f}", False, "waiting on lookback_rates")
            continue
        row = table.loc[g]
        if isinstance(row, pd.DataFrame):          # the group appears more than once
            row = row.iloc[0]
        have = {f: as_number(row[f]) for f in fields}
        owners_ok = have["owners"] == want["owners"]
        record(f"{g}: owners", owners_ok,
               f"expected {want['owners']}, got {show(row['owners'])}. {LOOKBACK_HINTS['owners']}")
        for back in ("back_1y", "back_2y"):
            record(f"{g}: {back}", have[back] == want[back],
                   f"expected {want[back]}, got {show(row[back])}. "
                   + (LOOKBACK_HINTS[back] if owners_ok else "Fix the owners in this group first."))
        for back, rate in (("back_1y", "rate_1y"), ("back_2y", "rate_2y")):
            target = want[back] / want["owners"]
            counts_ok = owners_ok and have[back] == want[back]
            record(f"{g}: {rate}", have[rate] is not None and abs(have[rate] - target) < 1e-9,
                   f"expected {want[back]} of {want['owners']} = {target:.4f}, got {show(row[rate])}. "
                   + (LOOKBACK_HINTS["rate"] if counts_ok else "It follows from the counts above."))

    report(results, "Your function matches 3.4 item 8 on the practice data. "
                    "Now build the real owner wallet transactions.")


check_lookback()

### Now the real look-back

Build the real list of **owner wallet transactions**, as 3.4 item 8 defines them: the qualifying loads, charges, refunds and promotional redemptions the owner initiated (the memo says which channels and types count) that were not reversed, dated on the day they happened. Build it once, from all the records through June 30, 2026. A reversal dated after June 30, 2024 still removes the load it points to. A login or a letter response is contact for the dormancy question, but it is not a return here, so this is not the same set of rows you used in section 4.

Then add one column to your owner table, next to `last_wallet_txn`: each owner's last owner wallet transaction on or before June 30, 2024.

**How items 8 and 9 are graded.** On a good-faith method you can explain. If your approach is simpler than this one, you will not lose credit for it. You will get feedback on how to refine it.

In [ ]:
# 1. The owner wallet transactions, as 3.4 item 8 defines them, through June 30, 2026.
# real_txns = ledger[...][["driver_id", "activity_on"]].rename(columns={"driver_id": "owner"})

# 2. The new column in your owner table (4.1 step 5).
# owners["last_txn_by_lookback"] = ...

# 3. The four rates.
# real_rates = lookback_rates(real_txns)
# real_rates.round(4)

In [ ]:
# import matplotlib.pyplot as plt
# ax = real_rates[["rate_1y", "rate_2y"]].plot(kind="bar", rot=0, figsize=(7, 4.5))
# ax.set_xlabel("How long the owner had been quiet on June 30, 2024")
# ax.set_ylabel("Share that came back")
# ax.legend(["Within one year", "Within two years"])
# ax.set_title("A title that states the finding, not the topic")

## 7. Breakage (4.1 step 6; 3.4 item 9)

- **The base.** The wallet cash at June 30, 2026, on the date each transaction happened, held by owners in the states that exempt stored value (after you resolve each owner's jurisdiction under the memo's rules) whose balance is above zero and whose last owner wallet transaction was on or before June 30, 2023, which is 36 months or more before the cutoff.
- **The estimate.** Multiply the base by one minus the two-year rate for the quiet-3-years-or-more group from section 6. Use the rate across owners in every state, unrounded. Round only the final dollars, to the cent.

The base looks at every transaction through June 30, 2026, not only the ones up to the look-back date. In the practice data, A001 is an account in Alder that was quiet 3 years or more on June 30, 2024. It came back on October 14, 2024, so it is not in the base.

This is breakage, not escheatment. In the reporting states dormant cash goes to the state. In one of them the memo lets the company keep part of it. Counsel asked about the exempt states only, so leave that retained share out of this estimate. Write down what the estimate assumes while you build it: owners who are quiet now behave like owners who were quiet in 2024, and two years without a return counts as never coming back. The auditors will ask what would make it wrong.

**Practice first.** Alder and Birch are made-up practice states. In the practice data they exempt stored value. For the project, the counsel memo decides which states do.

In [ ]:
PRACTICE_EXEMPT = ['Alder', 'Birch']   # made-up practice states. For the project, the counsel memo decides.
BASE_QUIET_THROUGH = pd.Timestamp("2023-06-30")   # 36 months before the cutoff

# Step 1. Each account's last transaction on every date through June 30, 2026, not only up to the look-back.
po = practice_owners.copy()
po["last_txn"] = po["owner"].map(practice_txns.groupby("owner")["activity_on"].max())

# Step 2. The base: an exempt state, a balance above zero, and a last transaction on or before June 30, 2023.
po["in_base"] = (po["state"].isin(PRACTICE_EXEMPT) & (po["cash_cents"] > 0)
                 & (po["last_txn"] <= BASE_QUIET_THROUGH))
practice_base = int(po.loc[po["in_base"], "cash_cents"].sum())

# Step 3. The estimate: the base times one minus the quiet-3-years-or-more two-year rate, rounded once.
rate_2y = practice_rates.loc["quiet 3 years or more", "rate_2y"]   # unrounded
practice_estimate = int(round(practice_base * (1 - rate_2y)))
print(f"Base:     {int(po['in_base'].sum())} accounts, {dollars(practice_base)}")
print(f"Estimate: {dollars(practice_estimate)}, with the two-year rate {rate_2y:.4f}")

# Stopping the clock early: the one-year rate is lower, so the estimate comes out higher.
rate_1y = practice_rates.loc["quiet 3 years or more", "rate_1y"]
print(f"With the one-year rate {rate_1y:.4f}, it would be {dollars(int(round(practice_base * (1 - rate_1y))))}. "
      "That overstates breakage.")

**Your turn.** Write `breakage` the same way, then run the check. The check gives your function the practice two-year rate, so it tests the base and the estimate on their own.

In [ ]:
def breakage(owners, txns, exempt_states, rate_2y):
    """
    owners: one row per owner, with columns owner, state (the jurisdiction) and cash_cents.
    txns: one row per owner wallet transaction, with columns owner and activity_on.
    exempt_states: the states that exempt stored value.
    rate_2y: the two-year rate of the quiet 3 years or more group, unrounded.

    Return a dict: {"base_cents": ..., "estimate_cents": ...}
    """
    return None   # your code

In [ ]:
# ===== RUN THIS to check breakage on the practice data. You do not edit this cell. =====
EXPECTED_BREAKAGE = {'base_cents': 460374, 'estimate_cents': 425846, 'rate_2y': [3, 40]}
BREAKAGE_HINTS = {
    "base": "The base is the cash of accounts in the exempt states with a balance above zero whose last transaction, on every date through June 30, 2026, was on or before June 30, 2023.",
    "estimate": "The estimate is the base times one minus the rate, rounded to the cent once.",
    "rate_not_one_minus": "That is the share that comes back. Breakage is the share that does not, so multiply by one minus the rate.",
    "no_rate": "That is the base itself. Multiply it by one minus the rate."
}
PRACTICE_RATE_2Y = EXPECTED_BREAKAGE["rate_2y"][0] / EXPECTED_BREAKAGE["rate_2y"][1]


def as_number(v):
    """v as a float when it is a number, else None."""
    if isinstance(v, bool):
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if f != f else f


def show(v):
    x = as_number(v)
    if x is None:
        return repr(v)
    return f"{int(x)}" if x == int(x) else f"{x:.4f}"


def report(results, done_message):
    done = sum(r[1] == "done" for r in results)
    print(f"{done} of {len(results)} checks done\n")
    for name, status, hint in results:
        print(f"  {status:8s} {name}" + (f"   ({hint})" if hint else ""))
    if done == len(results):
        print("\n" + done_message)


def show_cents(v):
    x = as_number(v)
    return repr(v) if x is None else f"{round(x):,} cents ({dollars(round(x))})"


def check_breakage():
    results = []

    def record(name, ok, hint=""):
        results.append((name, "done" if ok else "not yet", "" if ok else hint))

    got, waiting = None, None
    try:
        got = breakage(practice_owners.copy(), practice_txns.copy(), PRACTICE_EXEMPT, PRACTICE_RATE_2Y)
        if got is None:
            waiting = "it returns None. Write it first."
        elif not (isinstance(got, dict) and {"base_cents", "estimate_cents"} <= set(got)):
            waiting = 'it should return a dict with "base_cents" and "estimate_cents"'
    except Exception as e:
        waiting = f"it raised {type(e).__name__}: {e}"
    record("breakage returns base_cents and estimate_cents", waiting is None, waiting or "")

    if waiting:
        for name in ("base_cents", "estimate_cents"):
            record(name, False, "waiting on breakage")
    else:
        base, est = as_number(got["base_cents"]), as_number(got["estimate_cents"])
        want_base, want_est = EXPECTED_BREAKAGE["base_cents"], EXPECTED_BREAKAGE["estimate_cents"]
        base_ok = base is not None and abs(base - want_base) < 0.5
        record("base_cents", base_ok,
               f"expected {want_base:,} cents ({dollars(want_base)}), got {show_cents(got['base_cents'])}. "
               + BREAKAGE_HINTS["base"])
        if not base_ok:
            why = "It follows from the base."
        elif est is not None and abs(est - base * PRACTICE_RATE_2Y) < 1:
            why = BREAKAGE_HINTS["rate_not_one_minus"]
        elif est is not None and abs(est - base) < 1:
            why = BREAKAGE_HINTS["no_rate"]
        else:
            why = BREAKAGE_HINTS["estimate"]
        record("estimate_cents", est is not None and abs(est - want_est) <= 1,
               f"expected {want_est:,} cents ({dollars(want_est)}), got {show_cents(got['estimate_cents'])}. " + why)

    report(results, "Your function matches 3.4 item 9 on the practice data. Now run it on the real owners.")


check_breakage()

### Now the real breakage

Use your owner table from sections 3 and 4, the real transactions and the real rate from section 6, and the exempt states from your rule table. Item 9 is graded the same way as item 8: on a good-faith method you can explain.

In [ ]:
# real_owners = owners[["driver_id", "jurisdiction", "cash_cents"]].rename(
#     columns={"driver_id": "owner", "jurisdiction": "state"})
# EXEMPT_STATES = ...   # from your rule table in section 1
# rate_2y = real_rates.loc["quiet 3 years or more", "rate_2y"]   # unrounded
# real_breakage = breakage(real_owners, real_txns, EXEMPT_STATES, rate_2y)
# breakage_base_cents = real_breakage["base_cents"]
# breakage_cents = real_breakage["estimate_cents"]

## 8. The numbers

Fill in every value from your work above, by variable, not by typing a number in. Dollar values go in as cents and print as dollars to the cent. Rates go in as decimals and print to four places. Items 4 and 5 also ask for tables: the reportable amounts by state. Put the by-state amounts in dollars (divide the cents by 100 as the last step) so they read in the units 3.4 asks for. Item 7 asks for the list itself: put the driver_ids that get a letter in `letter_list`, and the count in `numbers`.

Then run the last cell. It prints everything in 3.4 order and lists anything still missing.

In [ ]:
numbers = {
    "1   Wallet cash liability at June 30":                  None,   # cents
    "2   GL 2110 minus your rebuilt cash liability":         None,   # cents
    "2   Transactions that explain the difference":          None,   # count
    "2   Cause of the difference":                           None,   # one sentence
    "3   Promotional credit at June 30":                     None,   # cents
    "3   Ties to GL 2115":                                   None,   # "yes" or "no"
    "4   Dormant wallets":                                   None,   # count
    "4   Reportable wallet cash, total":                     None,   # cents
    "5   Dormant deposits":                                  None,   # count
    "5   Reportable deposits, total":                        None,   # cents
    "6   Reportable to Delaware, both property types":       None,   # cents
    "6   Why each group reports to Delaware":                None,   # a sentence per group
    "7   Due diligence letters this cycle, count":           None,   # count
    "8   Quiet 3 years or more, back within 1 year":         None,   # rate
    "8   Quiet 3 years or more, back within 2 years":        None,   # rate
    "8   Quiet 1 to 3 years, back within 1 year":            None,   # rate
    "8   Quiet 1 to 3 years, back within 2 years":           None,   # rate
    "9   Breakage base":                                     None,   # cents
    "9   Breakage estimate":                                 None,   # cents
}
UNITS = {"1": "cents", "2   GL": "cents", "3   Promotional": "cents", "4   Reportable": "cents",
         "5   Reportable": "cents", "6   Reportable": "cents", "8": "rate", "9": "cents"}
LETTER_COUNT = "7   Due diligence letters this cycle, count"

letter_list = None        # list: the driver_ids that get a letter this cycle          (item 7)
wallet_by_state = None    # table: state, dormant wallets, reportable dollars           (item 4)
deposit_by_state = None   # table: state, dormant deposits, reportable dollars          (item 5)

In [ ]:
import textwrap
from numbers import Integral, Real
from IPython.display import display

print(f"Project 2 numbers, data build {BUILD_ID}\n")
missing = []
for label, value in numbers.items():
    unit = next((u for prefix, u in UNITS.items() if label.startswith(prefix)), None)
    if value is None:
        missing.append(label)
        shown = "MISSING"
    elif unit == "cents" and isinstance(value, Real):
        shown = dollars(value)
    elif unit == "rate" and isinstance(value, Real):
        shown = f"{value:.4f}"
    elif isinstance(value, Integral) and not isinstance(value, bool):
        shown = f"{int(value):,}"
    else:
        shown = str(value)
    print(f"{label:52s} {shown}")

print("\n7   Due diligence letter list (driver_id)")
if letter_list is None:
    missing.append("7   Due diligence letter list")
    print("MISSING")
else:
    ids = sorted(int(i) for i in letter_list)
    note = ""
    if numbers.get(LETTER_COUNT) is not None and int(numbers[LETTER_COUNT]) != len(ids):
        note += "   <- not the count above"
    if len(set(ids)) != len(ids):
        note += "   <- the list repeats an owner; a letter goes to each owner once"
    print(f"{len(ids):,} owners{note}")
    print(textwrap.fill(", ".join(str(i) for i in ids), width=100))

for name, table in [("4   Reportable wallet cash by state", wallet_by_state),
                    ("5   Reportable deposits by state", deposit_by_state)]:
    print(f"\n{name}")
    if table is None:
        missing.append(name)
        print("MISSING")
    else:
        display(table)

print("\nStill missing:" if missing else "\nEvery 3.4 number is present.")
for label in missing:
    print("  ", label.strip())

## 9. Export

Run every cell from the top (in Colab, Runtime > Run all) and check that the outputs are showing. This notebook, with its outputs, is your data support file. Download the notebook itself (in Colab, File > Download > Download .ipynb), or print it to PDF (File > Print, then Save as PDF). Upload it to the data support file question in the Project 2 quiz, with your deck and your one-pager.